# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/reetuparabat/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [1]:
"""
1) The contract, in plain words — Lane 2: Content Refresh Opportunity Scoring

a) One row means (my lane's scoring unit):
   One row = one content item, for one client (client_hash_id + content_hash_id),
   summarized over one calendar month. The row answers "how did this page perform
   in March 2026?" — not "how did it perform on one day." I build this row myself
   by aggregating the underlying daily fact rows for that content item within the
   month; the raw table underneath it is one row per content item, per client, per
   REPORT DAY, and I verify that daily grain in section 3 before trusting any
   aggregate built on top of it.

b) Tables I use:
   - fact_content_daily_performance (source of truth: daily GSC + GA4 metrics,
     grain = report_date x client_hash_id x content_hash_id) -- the ONLY table
     I aggregate from for this month's numbers.
   - dim_clients (context only: gsc_data_start / ga4_data_start, to check whether
     a client even has tracking in March 2026 before I count their rows as "no
     traffic").
   - dim_content (context only: joins content metadata like content_type,
     word_count for later feature work -- not used for the 5 features below).

c) Time window:
   month=2026-03 (a mid-panel month, per the assignment's warning), read directly
   from the fact table's month= partition. This is NOT the final month (2026-06)
   and NOT the _sample table -- that table is exactly the sealed final month and
   is off-limits for developing label logic.

d) What I'd predict or rank (label / proxy):
   A binary "needs review" proxy, same spirit as the starter CSV's
   is_declining_label: within a client-content pair's month, compare GSC
   impressions in the most recent ~30 days of the window against the ~30 days
   before that (spilling a few days into the prior partition, same trick notebook
   03 uses). >20% drop -> declining -> higher refresh priority. I build this proxy
   in section 3's trap experiment; the honest capstone version of this label will
   be a genuinely FUTURE outcome (this month's features -> next month's decline),
   not a same-window proxy -- I'm using the same-window version here only because
   it's fast to compute for the leakage demonstration.

e) One thing I deliberately exclude:
   GA4 engagement columns (ga4_sessions, ga4_engaged_sessions, etc.) for any
   row where ga4_data_available = FALSE. Those rows are zero-filled before a
   client's GA4 tracking started -- a zero there means "not measured," not "no
   engagement." I exclude them from GA4-based features rather than counting them
   as real zeros (verified with IS TRUE in section 3, query 3).
"""

'\n1) The contract, in plain words — Lane 2: Content Refresh Opportunity Scoring\n\na) One row means (my lane\'s scoring unit):\n   One row = one content item, for one client (client_hash_id + content_hash_id),\n   summarized over one calendar month. The row answers "how did this page perform\n   in March 2026?" — not "how did it perform on one day." I build this row myself\n   by aggregating the underlying daily fact rows for that content item within the\n   month; the raw table underneath it is one row per content item, per client, per\n   REPORT DAY, and I verify that daily grain in section 3 before trusting any\n   aggregate built on top of it.\n\nb) Tables I use:\n   - fact_content_daily_performance (source of truth: daily GSC + GA4 metrics,\n     grain = report_date x client_hash_id x content_hash_id) -- the ONLY table\n     I aggregate from for this month\'s numbers.\n   - dim_clients (context only: gsc_data_start / ga4_data_start, to check whether\n     a client even has tracki

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [11]:
"""
2) Field classification -- fact_content_daily_performance + dim_clients

FEATURE (knowable before the decision moment, safe to use):
  - gsc_impressions        -- daily GSC impressions, already observed history
  - gsc_clicks              -- daily GSC clicks, already observed history
  - gsc_avg_position        -- daily average search position, already observed
  - ga4_sessions / ga4_engaged_sessions -- daily GA4 activity, but ONLY on rows
    where ga4_data_available IS TRUE (see excluded, below)
  - report_date             -- used to build the month aggregation windows,
    not fed to the model directly as a raw date"""
"""
LABEL / PROXY (the thing predicted, or what it's computed from -- never a feature):
  - the last30-vs-prev30 impression comparison built in section 3 (my "declining"
    proxy) and anything computed directly from it. In the trap experiment this is
    the column I add on purpose and then delete."""
"""
CONTEXT (for grouping, joining, splitting -- never learned from):
  - client_hash_id, content_hash_id  -- pseudonymous join/group keys
  - dim_clients.gsc_data_start, ga4_data_start -- used to decide whether a
    client-month is even valid, not fed to the model"""
"""
EXCLUDED (private, product-decision, or future information -- with a why):
  - ga4_data_available = FALSE rows' GA4 columns -- excluded from GA4 features
    (zero-filled placeholder, not a true zero; see 1e)
  - any FlyRank product-decision field (health_score, priority_score,
    action_type, refresh flags) -- these are not shipped in the warehouse release
    at all, so there is nothing to strip, but I'm naming the rule so I don't
    accidentally rebuild one from raw signals and call it a feature
  - fact_content_daily_performance_sample -- the sealed final month; excluded from
    all iteration and label development, reserved as a held-out test partition
"""


"\nEXCLUDED (private, product-decision, or future information -- with a why):\n  - ga4_data_available = FALSE rows' GA4 columns -- excluded from GA4 features\n    (zero-filled placeholder, not a true zero; see 1e)\n  - any FlyRank product-decision field (health_score, priority_score,\n    action_type, refresh flags) -- these are not shipped in the warehouse release\n    at all, so there is nothing to strip, but I'm naming the rule so I don't\n    accidentally rebuild one from raw signals and call it a feature\n  - fact_content_daily_performance_sample -- the sealed final month; excluded from\n    all iteration and label development, reserved as a held-out test partition\n"

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [12]:
"""
Five features -- available when?

1. impressions_month -- sum of daily GSC impressions inside March 2026.
   Available when? Known the moment the month closes -- already-observed history.

2. clicks_month -- sum of daily GSC clicks inside March 2026.
   Available when? Same reasoning: fully observed history as of month-end.

3. avg_position_month -- mean daily gsc_avg_position, over days with impressions
   only (0/no-data days excluded, per the "avg_position = 0 means no data" gotcha).
   Available when? Computed only from days already in the past.

4. days_with_impressions -- count of days in the month with >= 1 impression.
   Available when? A tally of already-observed days.

5. ctr_month -- clicks_month / impressions_month x 100.
   Available when? Derived entirely from features 1 and 2, both already observed.
"""

'\nFive features -- available when?\n\n1. impressions_month -- sum of daily GSC impressions inside March 2026.\n   Available when? Known the moment the month closes -- already-observed history.\n\n2. clicks_month -- sum of daily GSC clicks inside March 2026.\n   Available when? Same reasoning: fully observed history as of month-end.\n\n3. avg_position_month -- mean daily gsc_avg_position, over days with impressions\n   only (0/no-data days excluded, per the "avg_position = 0 means no data" gotcha).\n   Available when? Computed only from days already in the past.\n\n4. days_with_impressions -- count of days in the month with >= 1 impression.\n   Available when? A tally of already-observed days.\n\n5. ctr_month -- clicks_month / impressions_month x 100.\n   Available when? Derived entirely from features 1 and 2, both already observed.\n'

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [13]:
"""
Named limitation: unbalanced panel + GSC-only early history.

History depth differs wildly per client (some have 17 months, some only a few),
and rows before a client's ga4_data_start are zero-filled with
ga4_data_available = FALSE. Query 3 above shows exactly what fraction of March
2026 survives that filter. Any feature or count built on this month without
first checking dim_clients.gsc_data_start / ga4_data_start risks treating
"tracking hadn't started yet" as "this page gets no traffic" -- a client whose
tracking begins March 15 doesn't have low March activity, it has no March data
at all for the first two weeks.
"""

'\nNamed limitation: unbalanced panel + GSC-only early history.\n\nHistory depth differs wildly per client (some have 17 months, some only a few),\nand rows before a client\'s ga4_data_start are zero-filled with\nga4_data_available = FALSE. Query 3 above shows exactly what fraction of March\n2026 survives that filter. Any feature or count built on this month without\nfirst checking dim_clients.gsc_data_start / ga4_data_start risks treating\n"tracking hadn\'t started yet" as "this page gets no traffic" -- a client whose\ntracking begins March 15 doesn\'t have low March activity, it has no March data\nat all for the first two weeks.\n'

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.